In [1]:
%%capture
!pip install pyspark
!pip install findspark
!pip install pyngrok

In [2]:
import findspark
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType, LongType

findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder \
        .appName('testColab') \
        .getOrCreate()

In [5]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyngrok import ngrok

# 1. Force Spark to stay on 4040
spark = SparkSession.builder \
    .appName('testColab') \
    .config("spark.ui.port", "4040") \
    .getOrCreate()

# 2. Check where Spark ACTUALLY landed
actual_ui_url = spark.sparkContext.uiWebUrl
print(f"Internal Spark UI is at: {actual_ui_url}")

# 3. Connect ngrok to that specific port
# (Using 4040 here because we forced it above)
ui_port = 4040
public_url = ngrok.connect(ui_port).public_url
print(f" * Public Spark UI: {public_url}")

Internal Spark UI is at: http://2e1ed7ddd441:4040
 * Public Spark UI: https://postelementary-unactivated-zola.ngrok-free.dev


In [4]:
from pyngrok import ngrok, conf
import getpass

print("Enter your authtoken, which can be copied "
"from https://dashboard.ngrok.com/get-started/your-authtoken")
conf.get_default().auth_token = getpass.getpass()

ui_port = 4040
public_url = ngrok.connect(ui_port).public_url
print(f" * ngrok tunnel \"{public_url}\" -> \"http://127.0.0.1:{ui_port}\"")

Enter your authtoken, which can be copied from https://dashboard.ngrok.com/get-started/your-authtoken
··········


 * ngrok tunnel "https://postelementary-unactivated-zola.ngrok-free.dev" -> "http://127.0.0.1:4040"


In [6]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2021-01.parquet

--2026-03-09 20:53:47--  https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2021-01.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.239.38.83, 18.239.38.147, 18.239.38.181, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.239.38.83|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 308924937 (295M) [application/x-www-form-urlencoded]
Saving to: ‘fhvhv_tripdata_2021-01.parquet’

fhvhv_tripdata_2021 100%[===================>] 294.61M  47.3MB/s    in 3.9s    

2026-03-09 20:53:51 (75.8 MB/s) - ‘fhvhv_tripdata_2021-01.parquet’ saved [308924937/308924937]



In [7]:
!wc -l fhvhv_tripdata_2021-01.parquet

1006794 fhvhv_tripdata_2021-01.parquet


In [8]:
df = spark.read.option("header", "true").parquet('fhvhv_tripdata_2021-01.parquet')

In [9]:
df.head()

Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', originating_base_num='B02682', request_datetime=datetime.datetime(2021, 1, 1, 0, 28, 9), on_scene_datetime=datetime.datetime(2021, 1, 1, 0, 31, 42), pickup_datetime=datetime.datetime(2021, 1, 1, 0, 33, 44), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 49, 7), PULocationID=230, DOLocationID=166, trip_miles=5.26, trip_time=923, base_passenger_fare=22.28, tolls=0.0, bcf=0.67, sales_tax=1.98, congestion_surcharge=2.75, airport_fee=None, tips=0.0, driver_pay=14.99, shared_request_flag='N', shared_match_flag='N', access_a_ride_flag=' ', wav_request_flag='N', wav_match_flag='N')

In [10]:
df.show()

+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+------------------+----------------+--------------+
|hvfhs_license_num|dispatching_base_num|originating_base_num|   request_datetime|  on_scene_datetime|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|access_a_ride_flag|wav_request_flag|wav_match_flag|
+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+--

In [11]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('originating_base_num', StringType(), True), StructField('request_datetime', TimestampNTZType(), True), StructField('on_scene_datetime', TimestampNTZType(), True), StructField('pickup_datetime', TimestampNTZType(), True), StructField('dropoff_datetime', TimestampNTZType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('trip_miles', DoubleType(), True), StructField('trip_time', LongType(), True), StructField('base_passenger_fare', DoubleType(), True), StructField('tolls', DoubleType(), True), StructField('bcf', DoubleType(), True), StructField('sales_tax', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('airport_fee', DoubleType(), True), StructField('tips', DoubleType(), True), StructField('driver_pay', DoubleType(), True), StructField('shared_re

In [12]:
!head -n 1001 fhvhv_tripdata_2021-01.parquet > new_df.csv

In [13]:
# Becuase spark doesnt infer the datatypes, lets use pandas to get the datatypes and greate a schema

import pandas as pd

In [14]:
# Read just the first 1000 rows directly from the source parquet
df_pandas = pd.read_parquet('fhvhv_tripdata_2021-01.parquet', engine='pyarrow').head(1000)

# Check the types
print(df_pandas.dtypes)

hvfhs_license_num               object
dispatching_base_num            object
originating_base_num            object
request_datetime        datetime64[us]
on_scene_datetime       datetime64[us]
pickup_datetime         datetime64[us]
dropoff_datetime        datetime64[us]
PULocationID                     int64
DOLocationID                     int64
trip_miles                     float64
trip_time                        int64
base_passenger_fare            float64
tolls                          float64
bcf                            float64
sales_tax                      float64
congestion_surcharge           float64
airport_fee                    float64
tips                           float64
driver_pay                     float64
shared_request_flag             object
shared_match_flag               object
access_a_ride_flag              object
wav_request_flag                object
wav_match_flag                  object
dtype: object


In [15]:
spark.createDataFrame(df_pandas).schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('originating_base_num', StringType(), True), StructField('request_datetime', TimestampType(), True), StructField('on_scene_datetime', TimestampType(), True), StructField('pickup_datetime', TimestampType(), True), StructField('dropoff_datetime', TimestampType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('trip_miles', DoubleType(), True), StructField('trip_time', LongType(), True), StructField('base_passenger_fare', DoubleType(), True), StructField('tolls', DoubleType(), True), StructField('bcf', DoubleType(), True), StructField('sales_tax', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('airport_fee', DoubleType(), True), StructField('tips', DoubleType(), True), StructField('driver_pay', DoubleType(), True), StructField('shared_request_flag',

In [16]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType, LongType
from pyspark.sql import types

In [17]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('originating_base_num', types.StringType(), True),
    types.StructField('request_datetime', types.TimestampType(), True),
    types.StructField('on_scene_datetime', types.TimestampType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    # IDs are INT64 in Parquet -> LongType in Spark
    types.StructField('PULocationID', types.LongType(), True),
    types.StructField('DOLocationID', types.LongType(), True),
    # Distance and Time
    types.StructField('trip_miles', types.DoubleType(), True),
    types.StructField('trip_time', types.LongType(), True),
    # All monetary values must be DoubleType to handle decimals
    types.StructField('base_passenger_fare', types.DoubleType(), True),
    types.StructField('tolls', types.DoubleType(), True),
    types.StructField('bcf', types.DoubleType(), True),
    types.StructField('sales_tax', types.DoubleType(), True),
    types.StructField('congestion_surcharge', types.DoubleType(), True),
    types.StructField('airport_fee', types.DoubleType(), True),
    types.StructField('tips', types.DoubleType(), True),
    types.StructField('driver_pay', types.DoubleType(), True),
    # Flags remain Strings
    types.StructField('shared_request_flag', types.StringType(), True),
    types.StructField('shared_match_flag', types.StringType(), True),
    types.StructField('access_a_ride_flag', types.StringType(), True),
    types.StructField('wav_request_flag', types.StringType(), True),
    types.StructField('wav_match_flag', types.StringType(), True)
])

In [18]:
# Use schema to read the file

df = spark.read.option("header", "true").schema(schema).parquet('fhvhv_tripdata_2021-01.parquet')

In [19]:
df.head()

Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', originating_base_num='B02682', request_datetime=datetime.datetime(2021, 1, 1, 0, 28, 9), on_scene_datetime=datetime.datetime(2021, 1, 1, 0, 31, 42), pickup_datetime=datetime.datetime(2021, 1, 1, 0, 33, 44), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 49, 7), PULocationID=230, DOLocationID=166, trip_miles=5.26, trip_time=923, base_passenger_fare=22.28, tolls=0.0, bcf=0.67, sales_tax=1.98, congestion_surcharge=2.75, airport_fee=None, tips=0.0, driver_pay=14.99, shared_request_flag='N', shared_match_flag='N', access_a_ride_flag=' ', wav_request_flag='N', wav_match_flag='N')

In [20]:
df = df.repartition(24)

In [21]:
df.write.parquet('fhvhv/2021/01/')

In [22]:
spark.version

'4.0.2'

 # Task

In [34]:
from pyspark.sql import functions as F

In [36]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-09 21:54:04--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.239.38.163, 18.239.38.147, 18.239.38.83, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.239.38.163|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet.1’

yellow_tripdata_202 100%[===================>]  67.84M   395MB/s    in 0.2s    

2026-03-09 21:54:04 (395 MB/s) - ‘yellow_tripdata_2025-11.parquet.1’ saved [71134255/71134255]



In [37]:
# Use schema defined above to read file

yellow = spark.read.option("header", "true").parquet('yellow_tripdata_2025-11.parquet')

In [38]:
yellow = yellow.repartition(4)

In [39]:
yellow.write.parquet('yellow/2021/')

In [41]:
yellow.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [44]:
#Select column and filter

yellow.select('tpep_pickup_datetime', 'tpep_pickup_datetime', 'PULocationID', 'DOLocationID').filter(F.col('PULOCATIONID') == 262)

DataFrame[tpep_pickup_datetime: timestamp_ntz, tpep_pickup_datetime: timestamp_ntz, PULocationID: int, DOLocationID: int]

In [43]:
yellow.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2025-11-07 15:04:17|  2025-11-07 15:39:15|              1|          7.3|         1|                 N|         262|    

In [46]:
# How many taxi trips were there on the 15th of November?

# 1. Convert the timestamp to a date and filter
# 2. Count the resulting rows
trip_count = yellow.filter(F.to_date(yellow.tpep_pickup_datetime) == "2025-11-15").count()

print(f"Number of trips on Nov 15th: {trip_count}")

Number of trips on Nov 15th: 162604


In [48]:
# What is the length of the longest trip in the dataset in hours?

# 1. Calculate duration in seconds
# 2. Convert to hours
# 3. Find the maximum value
df_with_duration = yellow.withColumn(
    'duration_hours',
    (F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime')) / 3600
)

longest_trip = df_with_duration.select(F.max('duration_hours')).collect()[0][0]

print(f"The longest trip was {longest_trip:.2f} hours.")

The longest trip was 90.65 hours.


In [50]:
# 1. Group by the Pickup Location ID
# 2. Count the occurrences
# 3. Sort by count in ascending order (smallest first)
# 4. Grab the top result
least_frequent_df = yellow.groupBy("PULocationID") \
                                .count() \
                                .orderBy("count", ascending=True)

least_frequent_df.show(5)

# To get just the ID as a variable:
least_id = least_frequent_df.first()['PULocationID']
print(f"The PULocationID with the least trips is: {least_id}")

+------------+-----+
|PULocationID|count|
+------------+-----+
|          84|    1|
|         105|    1|
|           5|    1|
|         187|    3|
|         204|    4|
+------------+-----+
only showing top 5 rows
The PULocationID with the least trips is: 84


In [51]:
# Pickup location

!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-09 22:14:40--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.239.38.163, 18.239.38.83, 18.239.38.147, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.239.38.163|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-09 22:14:40 (244 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [52]:
# Load the dataset

lookup = spark.read \
    .option("header", "true") \
    .csv('taxi_zone_lookup.csv')


In [53]:
# repartition to 4
lookup = lookup.repartition(4)

In [54]:
lookup.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|       158|    Manhattan|Meatpacking/West ...| Yellow Zone|
|        28|       Queens|Briarwood/Jamaica...|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|       219|       Queens|Springfield Garde...|   Boro Zone|
|       255|     Brooklyn|Williamsburg (Nor...|   Boro Zone|
|        75|    Manhattan|   East Harlem South|   Boro Zone|
|       254|        Bronx|Williamsbridge/Ol...|   Boro Zone|
|        32|        Bronx|           Bronxdale|   Boro Zone|
|        87|    Manhattan|Financial Distric...| Yellow Zone|
|        33|     Brooklyn|    Brooklyn Heights|   Boro Zone|
|        76|     Brooklyn|       East New York|   Boro Zone|
|        18|        Bronx|        Bedford Park|   Boro Zone|
|       195|     Brookly

In [55]:
# Join the datasets
# We join on PULocationID from the trips and LocationID from the lookup
joined_df = yellow.join(lookup, yellow.PULocationID == lookup.LocationID)

In [56]:
# Group by the 'Zone' name and count the occurrences
# Then sort by 'count' in ascending order to find the least frequent
least_frequent_zone = joined_df.groupBy("Zone") \
    .count() \
    .orderBy("count", ascending=True)

# Show the top results
least_frequent_zone.show(5, truncate=False)

# To get the name of the very first zone as a string:
result_zone = least_frequent_zone.first()['Zone']
print(f"The least frequent pickup zone is: {result_zone}")

+---------------------------------------------+-----+
|Zone                                         |count|
+---------------------------------------------+-----+
|Governor's Island/Ellis Island/Liberty Island|1    |
|Eltingville/Annadale/Prince's Bay            |1    |
|Arden Heights                                |1    |
|Port Richmond                                |3    |
|Rossville/Woodrow                            |4    |
+---------------------------------------------+-----+
only showing top 5 rows
The least frequent pickup zone is: Governor's Island/Ellis Island/Liberty Island
